In [ ]:
%pip install -q datasets pillow matplotlib pandas

In [ ]:
# Importando Biliotecas
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Carregando o dataset
dataset = load_dataset("huggan/wikiart")

In [ ]:
# Verificando se deu certo o download

print(dataset)

print(dataset["train"].features)

In [ ]:
# Guardando o nome dos estilos
style_feature = dataset["train"].features["style"]
style_names = style_feature.names

print(f"Quantidade de estilos: {len(style_names)}")

for i, style in enumerate(style_names):
    print(i, "-", style)

In [ ]:
# Quantas imagens existem em cada estilo
from collections import Counter

style_ids = dataset["train"]["style"]
contagem = Counter(style_ids)

for style_id, quantidade in contagem.most_common():
    print(f"{style_names[style_id]}: {quantidade}")

In [ ]:
# Transformando em tabela
dados_estilos = []

for style_id, quantidade in contagem.items():
    dados_estilos.append({
        "Estilo": style_names[style_id],
        "Quantidade": quantidade
    })

df_estilos = pd.DataFrame(dados_estilos)

df_estilos = df_estilos.sort_values(
    by="Quantidade",
    ascending=False
).reset_index(drop=True)

df_estilos

In [ ]:
# Gráfico
plt.figure(figsize=(10, 9))

plt.barh(
    df_estilos["Estilo"],
    df_estilos["Quantidade"]
)

plt.xlabel("Quantidade de imagens")
plt.ylabel("Estilo artístico")
plt.title("Distribuição de imagens por estilo no WikiArt")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Criando uma tabela apenas com estilo e artista
train = dataset["train"]

style_names = train.features["style"].names
artist_names = train.features["artist"].names

df_meta = pd.DataFrame({
    "style_id": train["style"],
    "artist_id": train["artist"]
})

df_meta["Estilo"] = df_meta["style_id"].map(lambda x: style_names[x])
df_meta["Artista"] = df_meta["artist_id"].map(lambda x: artist_names[x])

df_meta.head()


In [ ]:
# Quantos artistas existem em cada movimento?
resumo_artistas = (
    df_meta.groupby("Estilo")
    .agg(
        Quantidade_Obras=("Artista", "size"),
        Artistas_Distintos=("Artista", "nunique")
    )
    .reset_index()
    .sort_values("Quantidade_Obras", ascending=False)
)

resumo_artistas

In [ ]:
# Quantas obras de cada artista existem dentro de cada movimento?
artistas_por_estilo = (
    df_meta.groupby(["Estilo", "Artista"])
    .size()
    .reset_index(name="Quantidade")
    .sort_values(
        ["Estilo", "Quantidade"],
        ascending=[True, False]
    )
)

artistas_por_estilo.head(20)

In [ ]:
# Em porcentagem
total_por_estilo = (
    df_meta.groupby("Estilo")
    .size()
    .to_dict()
)

artistas_por_estilo["Percentual"] = artistas_por_estilo.apply(
    lambda linha:
        100 * linha["Quantidade"] / total_por_estilo[linha["Estilo"]],
    axis=1
).round(2)

artistas_por_estilo["Percentual"] = (
    artistas_por_estilo["Percentual"]
    .map(lambda x: f"{x:.2f}%")
)

artistas_por_estilo.head(20)

In [ ]:
# Descobrindo o artista mais frequente de cada movimento
artista_mais_frequente = (
    artistas_por_estilo
    .sort_values(["Estilo", "Quantidade"], ascending=[True, False])
    .groupby("Estilo")
    .first()
    .reset_index()
)

artista_mais_frequente[
    ["Estilo", "Artista", "Quantidade", "Percentual"]
]

In [ ]:
#Separando por generos
genre_names = train.features["genre"].names

df_meta["genre_id"] = train["genre"]
df_meta["Genero"] = df_meta["genre_id"].map(lambda x: genre_names[x])

df_meta.head()

generos_por_estilo = (
    df_meta.groupby(["Estilo", "Genero"])
    .size()
    .reset_index(name="Quantidade")
)

generos_por_estilo.head(20)

total_por_estilo = (
    df_meta.groupby("Estilo")
    .size()
    .to_dict()
)

generos_por_estilo["Percentual"] = generos_por_estilo.apply(
    lambda linha:
        100 * linha["Quantidade"] / total_por_estilo[linha["Estilo"]],
    axis=1
).round(2)

generos_por_estilo.head(20)

tabela_generos = pd.crosstab(
    df_meta["Estilo"],
    df_meta["Genero"],
    normalize="index"
) * 100

tabela_generos = tabela_generos.round(2)

tabela_generos

In [ ]:
# Aplicando filtro nos estilos selecionados
estilos_selecionados = [
    "Early_Renaissance",
    "High_Renaissance",
    "Northern_Renaissance",
    "Mannerism_Late_Renaissance",
    "Baroque",
    "Impressionism",
    "Cubism"
]

df_selecionados = df_meta[
    df_meta["Estilo"].isin(estilos_selecionados)
].copy()

df_selecionados.head()

df_selecionados["Estilo"].value_counts()

In [ ]:
#Juntando os estilos  de renascimento em uma macro renascimento
estilos_renascimento = [
    "Early_Renaissance",
    "High_Renaissance",
    "Northern_Renaissance",
    "Mannerism_Late_Renaissance"
]

df_selecionados["Grupo"] = df_selecionados["Estilo"].apply(
    lambda estilo: "Renaissance"
    if estilo in estilos_renascimento
    else estilo
)

df_selecionados["Grupo"].value_counts()

In [ ]:
artistas_selecionados = (
    df_selecionados
    .groupby(["Estilo", "Artista"])
    .size()
    .reset_index(name="Quantidade")
)

totais_estilo = (
    df_selecionados
    .groupby("Estilo")
    .size()
    .to_dict()
)

artistas_selecionados["Percentual"] = artistas_selecionados.apply(
    lambda linha:
        100 * linha["Quantidade"] / totais_estilo[linha["Estilo"]],
    axis=1
).round(2)

artistas_selecionados[
    artistas_selecionados["Estilo"] == "Cubism"
].sort_values(
    "Quantidade",
    ascending=False
).head(10)